# Best 3-Pair Merge Search
Evaluates every valid combination of three layer-index pairs and saves results to `results/{task}/best_3_pairs/`.  
Designed to run on **Google Colab** with CUDA, but falls back to CPU automatically.

In [ ]:
# ── Colab setup (skip if running locally) ────────────────────────────────────
import os

IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ
if IN_COLAB:
    # Clone the repo if not already present
    if not os.path.isdir('MME2'):
        !git clone https://github.com/YOUR_USERNAME/MME2.git   # <-- change URL
    os.chdir('MME2')
    !pip install -q -r requirements.txt

print('Working directory:', os.getcwd())

In [ ]:
import itertools
import json
from copy import deepcopy
from pathlib import Path

import torch
from transformers import CLIPModel, CLIPProcessor
from transformers import CLIPVisionModel

from src.eff_merge import *
from src.dataset_utils import *
from src.load_evaluate_utils import *

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

In [ ]:
# ── User-configurable section ─────────────────────────────────────────────────

TASKS        = ["MNIST", "SVHN", "EuroSAT", "Cars"]  # tasks to evaluate
SV_PORTION   = 2      # singular-value denominator k
WHITENING    = True   # apply whitening in TSV merge
LOOP         = False  # looped mode (model depth doesn't shrink)

RESULTS_BASE = Path('results')

# ── Pair generation ───────────────────────────────────────────────────────────
# index i means merging layers (i, i+1); valid range 0..10.
# Consecutive pairs [i, i+1] are excluded (would fail validation).
LAYER_INDICES = list(range(11))  # 0 to 10 inclusive

ALL_3_COMBOS = [
    list(combo)
    for combo in itertools.combinations(LAYER_INDICES, 3)
    if combo[1] != combo[0] + 1 and combo[2] != combo[1] + 1
]

print(f'Total pair combinations to evaluate: {len(ALL_3_COMBOS)}')
print('Combinations:', ALL_3_COMBOS)

In [ ]:
# ── Shared pretrained model (loaded once) ─────────────────────────────────────
pt_model     = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(DEVICE)
pt_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
print('Pretrained CLIP loaded.')

In [ ]:
# ── Main experiment loop ──────────────────────────────────────────────────────

all_results = {}  # task -> {config_key -> acc}

for task in TASKS:
    print('=' * 60)
    print(f'Task: {task}')
    print('=' * 60)

    # -- Dataset & text embeddings (once per task) ----------------------------
    ds_list              = [dataset_mapping[task]['dataset_id']]
    data_list, lab_list  = load_dataset_tasks(ds_list)
    data_load, lab_map, inv_lab_map = create_dataloader_from_list(data_list)
    temp_list            = [dataset_mapping[task]['template']]

    text_embeds, label_per_template, template_labels, total_num_templates = generate_text_embeddings(
        lab_list, temp_list,
        processor=pt_processor,
        model=pt_model,
        device=DEVICE
    )

    res_acc = {}

    # -- Iterate over all combination of 3 pairs -------------------------------
    for layer_indices in ALL_3_COMBOS:
        pairs_str = ', '.join(f'({idx},{idx+1})' for idx in layer_indices)
        print(f'  Merging pairs: {pairs_str}')

        ft_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
        ft_vision_model = CLIPVisionModel.from_pretrained(dataset_mapping[task]['model_id'])
        ft_model.vision_model.load_state_dict(ft_vision_model.vision_model.state_dict())
        ft_model = ft_model.to(DEVICE)

        merge_clip_blocks(
            ft_model,
            layer_idx=layer_indices,
            op_matrix=tsv_merge,
            op_vector=ft_average,
            k=SV_PORTION,
            whitening=WHITENING,
            loop=LOOP
        )

        acc = evaluate_model(
            ft_model, data_load,
            pt_processor, text_embeds, label_per_template, lab_map,
            device=DEVICE
        )

        config_key = str(layer_indices)
        print(f'    Accuracy: {acc:.5f}')
        res_acc[config_key] = acc

        # Free GPU memory after each combo
        del ft_model, ft_vision_model
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

    # -- Save results for this task -------------------------------------------
    out_dir = RESULTS_BASE / task / 'best_3_pairs'
    out_dir.mkdir(parents=True, exist_ok=True)
    results_path = out_dir / f'{task}_best_3_pairs.json'

    with open(results_path, 'w') as f:
        json.dump(res_acc, f, indent=4)

    all_results[task] = res_acc
    print(f'\n[OK] Results saved to {results_path}\n')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
print('=' * 60)
print('Summary of best 3 pairs combination per task:')
print('=' * 60)

for task in TASKS:
    results_path = RESULTS_BASE / task / 'best_3_pairs' / f'{task}_best_3_pairs.json'
    if results_path.exists():
        with open(results_path) as f:
            data = json.load(f)
        if data:
            best_key = max(data, key=data.get)
            best_acc = data[best_key]
            print(f'  {task:12s}: best combination = {best_key}  (acc = {best_acc:.5f})')
        else:
            print(f'  {task:12s}: no results found in file')
    else:
        print(f'  {task:12s}: no results file found')